In [0]:
# Start Reading the bronze tables
from pyspark.sql.functions import *

CATALOG = "workspace"
SCHEMA = "retail"

customers = spark.table(f"{CATALOG}.{SCHEMA}.bronze_customers")
products  = spark.table(f"{CATALOG}.{SCHEMA}.bronze_products")
orders    = spark.table(f"{CATALOG}.{SCHEMA}.bronze_orders")

In [0]:
silver_customers = (
    customers
        .dropDuplicates(["customer_id"])
        .filter(col("name").isNotNull())
        .withColumn("name", initcap(col("name")))
        .withColumn("email", lower(col("email")))
        .withColumn("processed_timestamp", current_timestamp())
)

In [0]:
silver_products = (
    products
        .dropDuplicates(["product_id"])
        .filter(col("price") > 0)
        .withColumn("category", initcap(col("category")))
        .withColumn("processed_timestamp", current_timestamp())
)

In [0]:
silver_orders = (
    orders
        .dropDuplicates(["order_id"])
        .filter(col("customer_id").isNotNull())
        .filter(col("product_id").isNotNull())
        .filter(col("quantity") > 0)
        .withColumn("order_date", to_date("order_date"))
        .withColumn("processed_timestamp", current_timestamp())
)

In [0]:
invalid_orders = (
    silver_orders
        .join(
            silver_customers,
            "customer_id",
            "left_anti"
        )
)

print("Invalid customer references:", invalid_orders.count())

In [0]:
(
    silver_customers.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_customers")
)

(
    silver_products.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_products")
)

(
    silver_orders.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_orders")
)

In [0]:
quality_report = [
    ("Customers", customers.count(), silver_customers.count()),
    ("Products", products.count(), silver_products.count()),
    ("Orders", orders.count(), silver_orders.count())
]

display(
    spark.createDataFrame(
        quality_report,
        ["table", "bronze_records", "silver_records"]
    )
)